In [ ]:
# Cell 1: install TrueRender demo dependencies, SAM 3, and patched TripoSR.
import os
import subprocess
import sys
from pathlib import Path

PYTHON = sys.executable
APP_DIR = Path.cwd()

# Auto-clone (or pull) when running in Colab from the default /content directory.
if not (APP_DIR / "app.py").exists():
    REPO_DIR = Path("/content/TrueRender")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "https://github.com/manny-serrano/TrueRender.git", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    APP_DIR = Path.cwd()

print("Notebook Python:", PYTHON)
print("Working directory:", APP_DIR)
assert (APP_DIR / "app.py").exists(), "Could not find app.py — repo clone may have failed."

base_install = [
    PYTHON, "-m", "pip", "install", "-q", "--upgrade",
    "fastapi", "uvicorn[standard]", "python-multipart", "trimesh", "pyngrok",
    "numpy==1.26.4", "opencv-python-headless==4.10.0.84", "pillow",
    "transformers==4.42.3",
]
print("Running:", " ".join(base_install))
subprocess.run(base_install, check=True)

# Install SAM 3.
os.chdir("/content")
if not os.path.exists("/content/sam3"):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/facebookresearch/sam3.git"], check=True)
os.chdir("/content/sam3")
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("SAM 3 ready")

# Install TripoSR clean mesh generator
TRIPOSR_REPO = "/content/TripoSR"
os.environ["TRIPOSR_REPO"] = TRIPOSR_REPO
os.chdir("/content")
if not os.path.exists(TRIPOSR_REPO):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/VAST-AI-Research/TripoSR", TRIPOSR_REPO], check=True)
os.chdir(TRIPOSR_REPO)

install_cmds = [
    [PYTHON, "-m", "pip", "install", "-q", "--upgrade", "setuptools", "wheel"],
    [PYTHON, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
]
for cmd in install_cmds:
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# Restore files before patching so this cell is idempotent even after a failed previous patch.
subprocess.run(["git", "checkout", "--", "run.py", "tsr/utils.py"], check=True)

# Patch TripoSR so --no-remove-bg does not import rembg/onnxruntime at module load time.
# We already provide a SAM-masked object composited onto gray, so rembg is unnecessary.
run_py = Path(TRIPOSR_REPO) / "run.py"
code = run_py.read_text()
code = code.replace(
    "import rembg\n",
    "# TRUERENDER_PATCH_LAZY_REMBG_RUN: rembg is imported lazily only when needed\nrembg = None\n",
    1,
)
expected = "if args.no_remove_bg:\n    rembg_session = None\nelse:\n    rembg_session = rembg.new_session()"
patched = "if args.no_remove_bg:\n    rembg_session = None\nelse:\n    import rembg\n    rembg_session = rembg.new_session()"
if expected not in code:
    raise RuntimeError("Could not find expected rembg_session block in TripoSR run.py")
code = code.replace(expected, patched, 1)

expected = "out_mesh_path = os.path.join(output_dir, str(i), f\"mesh.{args.model_save_format}\")"
patched = "out_mesh_path = os.path.join(output_dir, str(i), f\"mesh.{args.model_save_format}\")\nos.makedirs(os.path.dirname(out_mesh_path), exist_ok=True)"
if expected not in code:
    raise RuntimeError("Could not find expected out_mesh_path line in TripoSR run.py")
code = code.replace(expected, patched, 1)

run_py.write_text(code)
print("Patched TripoSR run.py for lazy rembg import and robust export directory creation")

utils_py = Path(TRIPOSR_REPO) / "tsr" / "utils.py"
code = utils_py.read_text()
code = code.replace(
    "import rembg\n",
    "# TRUERENDER_PATCH_LAZY_REMBG_UTILS: rembg is imported lazily only when remove_background is called\nrembg = None\n",
    1,
)
expected = "    if do_remove:\n        image = rembg.remove(image, session=rembg_session, **rembg_kwargs)"
patched = "    if do_remove:\n        import rembg\n        image = rembg.remove(image, session=rembg_session, **rembg_kwargs)"
if expected not in code:
    raise RuntimeError("Could not find expected remove_background block in TripoSR tsr/utils.py")
code = code.replace(expected, patched, 1)
utils_py.write_text(code)
print("Patched TripoSR tsr/utils.py to avoid rembg import when unused")

probe = subprocess.run(
    [PYTHON, "-c", "import sys; import tsr.utils; print(sys.executable); print('tsr.utils import ok')"],
    capture_output=True,
    text=True,
    check=True,
    cwd=TRIPOSR_REPO,
)
print("Child Python probe:\n" + probe.stdout)

os.chdir(APP_DIR)
print("TripoSR installed")
print("TrueRender demo setup complete")

In [ ]:
# Cell 2: launch FastAPI and expose it with ngrok.
import os
import threading
import time

import uvicorn
from pyngrok import ngrok

APP_DIR = os.getcwd()
os.environ.setdefault("TRIPOSR_REPO", "/content/TripoSR")

token = os.environ.get("NGROK_AUTHTOKEN")
if token:
    ngrok.set_auth_token(token)

def run_server():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, log_level="info")

if not globals().get("_truerender_server_thread") or not _truerender_server_thread.is_alive():
    _truerender_server_thread = threading.Thread(target=run_server, daemon=True)
    _truerender_server_thread.start()
    time.sleep(2)

public_url = ngrok.connect(8000, bind_tls=True).public_url
print("Open this URL in a new tab to use the demo")
print(public_url)